# Практика 4 Частотная область и многомасштабный анализ


## 0 Среда и воспроизводимость

Нужны `numpy`, `matplotlib`, `scipy`, `scikit-image` и `Pillow`. Значения изображений храним как `float64` в диапазоне `[0, 1]`; перед разностями не используем `uint8`.


In [ ]:
%matplotlib inline
import io
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from skimage import data, img_as_float
from skimage.metrics import peak_signal_noise_ratio
from skimage.transform import resize

np.random.seed(7)
plt.rcParams.update({"figure.figsize": (11, 4), "image.cmap": "gray", "axes.grid": False})

def show_grid(items, ncols=3, figsize=None):
    # items: последовательность (title, image, optional kwargs).
    n = len(items)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize or (4*ncols, 3.5*nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, item in zip(axes, items):
        title, image, *rest = item
        kwargs = rest[0] if rest else {}
        ax.imshow(image, **kwargs)
        ax.set_title(title)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    return fig

def fft_centered(image):
    return np.fft.fftshift(np.fft.fft2(image))

def spectrum_view(image):
    return np.log1p(np.abs(fft_centered(image)))

def psnr(reference, test):
    return peak_signal_noise_ratio(reference, np.clip(test, 0, 1), data_range=1.0)

IMAGE_PATH = "camera.png"

with Image.open(IMAGE_PATH) as image:
    camera = np.asarray(image.convert("L"), dtype=np.float64) / 255.0

camera = resize(camera, (256, 256), anti_aliasing=True)

print(camera.shape, camera.dtype, float(camera.min()), float(camera.max()))


In [ ]:
assert camera.shape == (256, 256)
assert camera.dtype.kind == "f"
assert 0 <= camera.min() <= camera.max() <= 1
show_grid([("Рабочее изображение", camera)])


## 1 Частотный детектив

Косинусная решётка с частотным вектором `(u, v)` создаёт две симметричные точки спектра: `(u, v)` и `(-u, -v)` относительно центра. Полосы на изображении перпендикулярны этому вектору.


In [ ]:
def cosine_grating(shape, u, v, amplitude=0.45, mean=0.5, phase=0.0):
    m, n = np.indices(shape)
    M, N = shape
    wave = np.cos(2*np.pi*(u*m/M + v*n/N) + phase)
    return mean + amplitude * wave

hidden_u, hidden_v = 15, -28
grating = cosine_grating((192, 192), hidden_u, hidden_v, phase=0.7)
show_grid([
    ("Неизвестная решётка", grating),
    ("log(1 + |F|)", spectrum_view(grating)),
], ncols=2, figsize=(10, 4))


### Задание 1 Найти частотный вектор

Реализуйте `dominant_offsets`:

1. вычислите центрированный спектр;
2. занулите круг радиуса `exclude_radius` вокруг центра, чтобы убрать DC-компонент;
3. верните смещения `n` самых ярких точек относительно центра в формате `(du, dv)`.


In [ ]:
def dominant_offsets(image, n=2, exclude_radius=5):
    # TODO: верните n смещений (du, dv) самых ярких пиков от центра спектра.
    raise NotImplementedError


In [ ]:
peaks = dominant_offsets(grating)
print("Найденные смещения:", peaks)
expected = {(hidden_u, hidden_v), (-hidden_u, -hidden_v)}
assert set(peaks) == expected, f"Ожидались {expected}, получено {set(peaks)}"


## 2 Удаление периодической помехи

Добавим к фотографии косую периодическую наводку. В пространстве она покрывает всю сцену, а в спектре концентрируется в двух локальных пиках. Значит, можно подавить только маленькие окрестности этих частот.


In [ ]:
noise_u, noise_v = 24, 37
periodic = cosine_grating(camera.shape, noise_u, noise_v, amplitude=0.13, mean=0.0, phase=0.2)
noisy = np.clip(camera + periodic, 0, 1)

show_grid([
    ("Чистое", camera),
    ("С периодической помехой", noisy),
    ("Спектр чистого", spectrum_view(camera)),
    ("Спектр с помехой", spectrum_view(noisy)),
], ncols=2, figsize=(10, 8))
print(f"PSNR зашумлённого: {psnr(camera, noisy):.2f} dB")


### Задание 2 Построить гладкий режекторный фильтр

Реализуйте `gaussian_notch_mask(shape, offsets, sigma)`.

- Маска равна 1 почти везде.
- В каждой точке из `offsets` относительно центра должна быть гладкая гауссова «дырка» до 0.
- Для действительного изображения подавляйте симметричную пару частот.

Затем умножьте центрированный спектр на маску и выполните обратное ДПФ. Не забудьте `ifftshift` и `.real`.


In [ ]:
def gaussian_notch_mask(shape, offsets, sigma=2.5):
    # TODO: постройте произведение гладких гауссовых режекторов.
    raise NotImplementedError


In [ ]:
offsets = [(noise_u, noise_v), (-noise_u, -noise_v)]
H_notch = gaussian_notch_mask(noisy.shape, offsets, sigma=2.5)

cy, cx = np.array(noisy.shape) // 2
assert H_notch.shape == noisy.shape
assert np.all((0 <= H_notch) & (H_notch <= 1))
assert H_notch[cy + noise_u, cx + noise_v] < 0.02
assert H_notch[cy - noise_u, cx - noise_v] < 0.02
assert H_notch[cy, cx] > 0.99

F_noisy = fft_centered(noisy)
filtered = np.fft.ifft2(np.fft.ifftshift(F_noisy * H_notch)).real
filtered = np.clip(filtered, 0, 1)
print(f"PSNR: noisy={psnr(camera, noisy):.2f} dB, filtered={psnr(camera, filtered):.2f} dB")
assert psnr(camera, filtered) > psnr(camera, noisy) + 8


In [ ]:
crop = np.s_[70:160, 70:180]
show_grid([
    ("Маска H", H_notch, {"vmin": 0, "vmax": 1}),
    ("До фильтра", noisy[crop]),
    ("После фильтра", filtered[crop]),
    ("Удалённый компонент", noisy[crop] - filtered[crop], {"cmap": "coolwarm", "vmin": -0.16, "vmax": 0.16}),
], ncols=2, figsize=(9, 8))


**Визуальная проверка.** Полосы должны исчезнуть, а реальные границы камеры и штатива — сохраниться. На карте удалённого компонента ожидается почти чистая синусоида. Если вместе с помехой исчезли детали, уменьшите `sigma`; если полосы остались — немного увеличьте.


## 3 Лапласова пирамида

Гауссова пирамида хранит всё более грубые копии. Лапласов уровень хранит то, что теряется при переходе к следующему масштабу:

$$L_l = G_l - \operatorname{expand}(G_{l+1}).$$

Поэтому значения `L_l` могут быть отрицательными, а восстановление выполняется снизу вверх сложением.


In [ ]:
def reduce2(image):
    target = ((image.shape[0] + 1)//2, (image.shape[1] + 1)//2)
    return resize(image, target, anti_aliasing=True, preserve_range=True)

def expand_to(image, shape):
    return resize(image, shape, anti_aliasing=False, preserve_range=True)

def gaussian_pyramid(image, levels=4):
    pyr = [np.asarray(image, dtype=float)]
    for _ in range(levels):
        pyr.append(reduce2(pyr[-1]))
    return pyr

G = gaussian_pyramid(camera, levels=4)
print([x.shape for x in G])


### Задание 3 Разложить и точно собрать изображение

Реализуйте две функции.

- `laplacian_from_gaussian(G)` возвращает `L0, ..., LK`; последний уровень равен `GK`.
- `reconstruct_laplacian(L)` собирает изображение снизу вверх.

Проверка должна быть строгой: максимальная абсолютная ошибка меньше `1e-10`. Такая точность возможна, потому что при разложении и сборке используется одна и та же операция `expand_to`.


In [ ]:
def laplacian_from_gaussian(G):
    # TODO: L_l = G_l - expand(G_{l+1}), последний уровень L_K = G_K.
    raise NotImplementedError

def reconstruct_laplacian(L):
    # TODO: начните с L_K и собирайте уровни снизу вверх.
    raise NotImplementedError


In [ ]:
L = laplacian_from_gaussian(G)
reconstructed = reconstruct_laplacian(L)
max_error = float(np.max(np.abs(camera - reconstructed)))
print("Формы L:", [x.shape for x in L])
print(f"max |I - I_rec| = {max_error:.3e}")
assert len(L) == len(G)
assert max_error < 1e-10


In [ ]:
gaussian_items = [(f"G{level}  {img.shape}", img, {"vmin": 0, "vmax": 1}) for level, img in enumerate(G)]
show_grid(gaussian_items, ncols=3, figsize=(11, 7))

lap_items = []
for level, img in enumerate(L[:-1]):
    scale = max(np.percentile(np.abs(img), 99), 1e-6)
    lap_items.append((f"L{level}  детали", img, {"cmap": "coolwarm", "vmin": -scale, "vmax": scale}))
lap_items.append((f"L{len(L)-1}  остаток", L[-1], {"vmin": 0, "vmax": 1}))
show_grid(lap_items, ncols=3, figsize=(11, 7))

show_grid([
    ("Исходное", camera),
    ("Восстановленное", reconstructed),
    ("Ошибка × 10^12", (camera-reconstructed)*1e12, {"cmap": "coolwarm"}),
], ncols=3, figsize=(12, 4))


**Визуальная проверка.** На `L0` ожидаются тонкие границы и текстуры, на более грубых уровнях — крупные контуры. Ошибка восстановления визуально должна быть однородной около машинного нуля.


## 4 JPEG

JPEG кодирует блоки 8×8 в DCT, квантует коэффициенты и затем сжимает серии нулей. Измерим две стороны компромисса:

- **bpp** — сколько бит файла приходится на пиксель;
- **PSNR** — насколько декодированное изображение близко к исходному.

Визуально отдельно ищем блочность 8×8 и звон около резких границ: одна скалярная метрика не гарантирует отсутствия локальных артефактов.


### Задание 4 Построить кривую размер качество

Реализуйте `jpeg_roundtrip(image, quality)` через `PIL.Image` и `io.BytesIO`. Функция должна вернуть декодированное изображение в `[0,1]` и число байт буфера. Затем найдите **минимальное** качество из сетки `5, 10, ..., 95`, при котором PSNR не ниже 32 dB.


In [ ]:
def jpeg_roundtrip(image, quality):
    # TODO: кодирование в BytesIO, декодирование в float [0, 1], возврат (decoded, size_bytes).
    raise NotImplementedError


In [ ]:
qualities = list(range(5, 100, 5))
records = []
decoded_by_q = {}
for q in qualities:
    decoded, nbytes = jpeg_roundtrip(camera, q)
    decoded_by_q[q] = decoded
    bpp = 8 * nbytes / camera.size
    records.append((q, bpp, psnr(camera, decoded)))

eligible = [r for r in records if r[2] >= 32]
best_q = min(r[0] for r in eligible)
print("Минимальное quality при PSNR >= 32 dB:", best_q)
print("quality | bpp | PSNR")
for q, bpp, score in records:
    if q in [5, 15, 30, 50, 75, 95] or q == best_q:
        print(f"{q:>7} | {bpp:4.2f} | {score:5.2f}")

assert decoded_by_q[50].shape == camera.shape
assert all(records[i][1] <= records[i+1][1] for i in range(len(records)-1))


In [ ]:
q_arr = np.array([r[0] for r in records])
bpp_arr = np.array([r[1] for r in records])
psnr_arr = np.array([r[2] for r in records])
plt.figure(figsize=(6, 4))
plt.plot(bpp_arr, psnr_arr, "o-")
for q in [5, best_q, 50, 95]:
    idx = qualities.index(q)
    plt.annotate(f"Q={q}", (bpp_arr[idx], psnr_arr[idx]), xytext=(5, 5), textcoords="offset points")
plt.xlabel("бит на пиксель")
plt.ylabel("PSNR, dB")
plt.title("JPEG: размер против качества")
plt.grid(alpha=0.3)
plt.show()

crop = np.s_[25:145, 70:200]
inspect_q = [95, 50, best_q, 10]
show_grid([
    (f"Q={q}, {psnr(camera, decoded_by_q[q]):.1f} dB", decoded_by_q[q][crop], {"vmin": 0, "vmax": 1})
    for q in inspect_q
], ncols=4, figsize=(14, 4))


**Визуальная проверка.** На низком качестве ищите сетку блоков 8×8 в гладких областях и ореолы около контрастных границ. Сравните `best_q` с более низким качеством: порог PSNR отвечает на численный вопрос, но окончательный выбор зависит от того, где находятся артефакты.

Почему увеличение `quality` даёт всё меньший прирост PSNR на каждый дополнительный бит? Почему маску сегментации нельзя хранить в JPEG даже при высоком качестве?
